In [35]:
import os
import bz2
import pickle
import urllib.request
import tarfile
import re
import requests
from nltk.tokenize import TreebankWordTokenizer

In [6]:
filename = "reviewdata.pickle.bz2"
if os.path.exists(filename):
    print(f"Using cached file {filename}")
    with bz2.BZ2File(filename, "r") as zipfile:
        data = pickle.load(zipfile)
    text_train, text_test, y_train, y_test = data
else:
    url = "https://cssbook.net/d/aclImdb_v1.tar.gz"
    print(f"Downloading from {url}")
    fn, _headers = urllib.request.urlretrieve(url, filename=None)
    t = tarfile.open(fn, mode="r:gz")
    text_train, text_test = [], []
    y_train, y_test = [], []
    for f in t.getmembers():
        m = re.match(r"aclImdb/(\w+)/(pos|neg)/", f.name)
        if not m:
            # skip folder names, other categories
            continue
        dataset, label = m.groups()
        text = t.extractfile(f).read().decode("utf-8")
        if dataset == "train":
            text_train.append(text)
            y_train.append(label)
        elif dataset == "test":
            text_test.append(text)
            y_test.append(label)
    data = text_train, text_test, y_train, y_test
    print(f"Saving to {filename}")
    with bz2.BZ2File(filename, "w") as zipfile:
        pickle.dump(data, zipfile)

Using cached file reviewdata.pickle.bz2


In [ ]:
### Retrieve 

In [27]:
poswords = "https://cssbook.net/d/positive.txt"
negwords = "https://cssbook.net/d/negative.txt"
pos = set(requests.get(poswords).text.split("\n"))
neg = set(requests.get(negwords).text.split("\n"))

In [29]:
pos.remove('')

In [30]:
neg.remove('')

In [31]:
sentimentdict = {word: +1 for word in pos}
sentimentdict.update({word: -1 for word in neg})

## Running Sentiment Analysis -- according to the book

In [50]:
scores = []
mytokenizer = TreebankWordTokenizer()
# For speed, we only take the first 100 reviews
for review in text_train[:100]:
    words = mytokenizer.tokenize(review)
    # we look up each word in the sentiment dict
    # and assign its value (with default 0)
    scores.append(sum(sentimentdict.get(word, 0) for word in words))
print(scores)

[-3, -4, 1, 3, -2, -7, -6, 9, 7, 7, 10, 5, -1, 2, 7, -4, 2, 21, 1, -1, 2, -3, -2, -11, -2, -3, -7, 2, 4, -22, 5, 4, 3, -5, -8, 1, -1, 0, 1, 8, 0, -4, 3, -7, -11, -6, 0, 3, -1, 0, 6, -1, -8, 7, -5, 2, 10, 5, 5, 1, 0, 7, 0, 0, 5, 1, -8, 4, 3, 18, 2, 0, -3, -2, 5, 0, -2, 1, 1, 12, -3, -4, -6, -2, 2, -7, -1, -10, -5, 3, 4, -3, -17, 1, -1, 7, -3, 4, 12, 3]


## What does `.get(word, 0)` do?

- `sentimentdict` is a dictionary mapping words to numbers (like `"great": 1` or `"boring": -1`).
- Sometimes a word we look up isn't in the dictionary.
- Using `sentimentdict[word]` on a missing word would cause a **KeyError**.
- `.get(word, 0)` avoids that error (similiarly to what we've seen before using a defaultdict)
- It returns:
  - the value if the word exists in the dictionary
  - `0` if the word does not exist
- This treats unknown words as neutral and keeps the program from crashing.

**“Give me the score for this word, or 0 if we don’t have it.”**

In [37]:
sentimentdict.get('amazing', 0)

1

In [43]:
scores = []

for review in text_train[:10]:
    # Step 1: tokenize the review into words
    words = mytokenizer.tokenize(review)

    # Step 2: start the score at 0
    score = 0

    # Step 3: loop through each word
    for word in words:
        # safely get the value from the dictionary
        # +1, -1, or 0 if the word isn't there
        value = sentimentdict.get(word, 0)
        print(f"the word its checking: {word}, with value: {value}")
        # add that value to the score
        score += value

    # Step 4: store the final score for this review
    scores.append(score)

print(scores)

the word its checking: I, with value: 0
the word its checking: rented, with value: 0
the word its checking: I, with value: 0
the word its checking: AM, with value: 0
the word its checking: CURIOUS-YELLOW, with value: 0
the word its checking: from, with value: 0
the word its checking: my, with value: 0
the word its checking: video, with value: 0
the word its checking: store, with value: 0
the word its checking: because, with value: 0
the word its checking: of, with value: 0
the word its checking: all, with value: 0
the word its checking: the, with value: 0
the word its checking: controversy, with value: 0
the word its checking: that, with value: 0
the word its checking: surrounded, with value: 0
the word its checking: it, with value: 0
the word its checking: when, with value: 0
the word its checking: it, with value: 0
the word its checking: was, with value: 0
the word its checking: first, with value: 0
the word its checking: released, with value: 0
the word its checking: in, with value:

## The same, but this time without .get but with defaultdict

- It automatically provides a **default value** when a key does not exist.
- If we use `defaultdict(int)`, the default value is `0`.
- This means we **don’t get a KeyError**, and we **don’t need `.get(word, 0)`**.
- Unknown words simply behave like neutral words with a score of `0`.

**“If the word isn’t in the dictionary, treat it as 0 automatically.”**


In [47]:
from collections import defaultdict

sentimentdict = defaultdict(int)

# Step 2: fill it with positive and negative words
for word in pos:
    sentimentdict[word] = 1

for word in neg:
    sentimentdict[word] = -1

    
scores = []

for review in text_train[:10]:
    # tokenize the review
    words = mytokenizer.tokenize(review)

    # start the score at zero
    score = 0

    # loop through each word
    for word in words:
        value = sentimentdict[word]
        score += value

    # store the final score
    scores.append(score)

print(scores)

[-3, -4, 1, 3, -2, -7, -6, 9, 7, 7]


## Code from the book

[-3, -4, 1, 3, -2, -7, -6, 9, 7, 7, 10, 5, -1, 2, 7, -4, 2, 21, 1, -1, 2, -3, -2, -11, -2, -3, -7, 2, 4, -22, 5, 4, 3, -5, -8, 1, -1, 0, 1, 8, 0, -4, 3, -7, -11, -6, 0, 3, -1, 0, 6, -1, -8, 7, -5, 2, 10, 5, 5, 1, 0, 7, 0, 0, 5, 1, -8, 4, 3, 18, 2, 0, -3, -2, 5, 0, -2, 1, 1, 12, -3, -4, -6, -2, 2, -7, -1, -10, -5, 3, 4, -3, -17, 1, -1, 7, -3, 4, 12, 3]
